# 08 — Coastal Gradient Experiment


This notebook builds a Lewes–Newhaven–Eastbourne coastal gradient experiment using Sentinel-5P scene catalogs from 07b, optional scene-level NO2 statistics from 07c, and Open-Meteo historical weather.

It writes a manifest, candidate scene-day table, gradient results tables, and a plot.


In [ ]:

SITES = [
    {"site_id": "EST", "site_name": "Eastbourne",   "lat": 50.7680,  "lon": 0.2900},
    {"site_id": "NHV", "site_name": "Newhaven ERF", "lat": 50.80208, "lon": 0.04961},
    {"site_id": "LWS", "site_name": "Lewes",        "lat": 50.8739,  "lon": 0.0110},
]
DATE_FROM = "2024-12-26"
DATE_TO   = "2025-01-05"
PRODUCT_KEY = "NO2"
TIMELINESS = "OFFL"
SCENE_CATALOG_DIR = "outputs/s5p"
SCENE_STATS_DIR   = "outputs/s5p_stack"
OUTPUT_DIR        = "outputs/gradient"
MAX_TIME_MISMATCH_HOURS = 3


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import json, math, hashlib
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()
print("UTC now:", datetime.now(timezone.utc).isoformat())


In [ ]:

catalog_rows = []
input_files = []
for site in SITES:
    p = Path(SCENE_CATALOG_DIR) / f"{site['site_id']}_{PRODUCT_KEY.lower()}_{TIMELINESS.lower()}_scene_catalog.csv"
    if p.exists():
        df = pd.read_csv(p)
        df["site_id"] = site["site_id"]
        df["site_name"] = site["site_name"]
        catalog_rows.append(df)
        input_files.append({"path": str(p), "sha256": sha256_file(p)})
    else:
        print("Missing catalog:", p)
cat = pd.concat(catalog_rows, ignore_index=True) if catalog_rows else pd.DataFrame()
if not cat.empty:
    cat["start_utc"] = pd.to_datetime(cat["start_utc"], utc=True, errors="coerce")
    cat["date"] = cat["start_utc"].dt.date.astype("string")
    cat = cat[(cat["start_utc"] >= pd.Timestamp(DATE_FROM, tz="UTC")) &
              (cat["start_utc"] <= pd.Timestamp(DATE_TO, tz="UTC") + pd.Timedelta(days=1))]
print("Scene catalog rows loaded:", len(cat))
display(cat.head(10))


In [ ]:

stats_rows = []
for site in SITES:
    p_csv = Path(SCENE_STATS_DIR) / f"{site['site_id']}_scene_level_no2_window_stats.csv"
    p_pq = Path(SCENE_STATS_DIR) / f"{site['site_id']}_scene_level_no2_window_stats.parquet"
    p = p_csv if p_csv.exists() else p_pq if p_pq.exists() else None
    if p is None:
        continue
    df = pd.read_csv(p) if p.suffix == ".csv" else pd.read_parquet(p)
    df["site_id"] = site["site_id"]
    df["site_name"] = site["site_name"]
    if "start_utc" in df.columns:
        df["start_utc"] = pd.to_datetime(df["start_utc"], utc=True, errors="coerce")
        df["date"] = df["start_utc"].dt.date.astype("string")
    stats_rows.append(df)
    input_files.append({"path": str(p), "sha256": sha256_file(p)})
stats = pd.concat(stats_rows, ignore_index=True) if stats_rows else pd.DataFrame()
print("Scene stats rows loaded:", len(stats))
display(stats.head(10))


In [ ]:

def fetch_openmeteo_hourly(lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "wind_speed_10m,wind_direction_10m,cloud_cover",
        "timezone": "UTC",
    }
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    j = r.json()
    h = j.get("hourly", {})
    return pd.DataFrame({
        "time_utc": pd.to_datetime(h.get("time", []), utc=True, errors="coerce"),
        "wind_speed_ms": h.get("wind_speed_10m", []),
        "wind_dir_deg": h.get("wind_direction_10m", []),
        "cloud_cover_pct": h.get("cloud_cover", []),
    })

weather_rows = []
for site in SITES:
    w = fetch_openmeteo_hourly(site["lat"], site["lon"], DATE_FROM, DATE_TO)
    w["site_id"] = site["site_id"]
    w["site_name"] = site["site_name"]
    weather_rows.append(w)
weather = pd.concat(weather_rows, ignore_index=True)
display(weather.head(10))


In [ ]:

def nearest_hour(ts, w):
    if w.empty or pd.isna(ts):
        return np.nan, np.nan, np.nan, pd.NaT
    idx = (w["time_utc"] - ts).abs().idxmin()
    row = w.loc[idx]
    dt_h = abs((row["time_utc"] - ts).total_seconds()) / 3600.0
    return row["wind_speed_ms"], row["wind_dir_deg"], row["cloud_cover_pct"], dt_h

if cat.empty:
    gradient = pd.DataFrame()
else:
    rows = []
    for _, r in cat.iterrows():
        w = weather[weather["site_id"] == r["site_id"]].copy()
        ws, wd, cc, dt_h = nearest_hour(r["start_utc"], w)
        if pd.notna(dt_h) and dt_h > MAX_TIME_MISMATCH_HOURS:
            continue
        out = {
            "date": str(r["date"]),
            "site_id": r["site_id"],
            "site_name": r["site_name"],
            "start_utc": r["start_utc"],
            "scene_name": r.get("name"),
            "product_id": r.get("product_id"),
            "wind_speed_ms": ws,
            "wind_dir_deg": wd,
            "cloud_cover_pct": cc,
            "has_scene_stats": False,
            "mean_no2": np.nan,
            "median_no2": np.nan,
            "p95_no2": np.nan,
        }
        if not stats.empty and "product_id" in stats.columns:
            m = stats[(stats["site_id"] == r["site_id"]) & (stats["product_id"].astype(str) == str(r.get("product_id")))]
            if not m.empty:
                row = m.iloc[0]
                out["has_scene_stats"] = True
                out["mean_no2"] = row.get("mean_no2", np.nan)
                out["median_no2"] = row.get("median_no2", np.nan)
                out["p95_no2"] = row.get("p95_no2", np.nan)
        rows.append(out)
    gradient = pd.DataFrame(rows)

distance_order = {"EST": 0, "NHV": 1, "LWS": 2}
if not gradient.empty:
    gradient["position_index"] = gradient["site_id"].map(distance_order)
    daily_summary = (
        gradient.groupby(["date", "site_id", "site_name", "position_index"], dropna=False)
        .agg(
            mean_wind_speed_ms=("wind_speed_ms", "mean"),
            mean_cloud_cover_pct=("cloud_cover_pct", "mean"),
            scenes=("product_id", "nunique"),
            has_scene_stats=("has_scene_stats", "max"),
            mean_no2=("mean_no2", "mean"),
            median_no2=("median_no2", "mean"),
        )
        .reset_index()
        .sort_values(["date", "position_index"])
    )
else:
    daily_summary = pd.DataFrame()

gradient_csv = Path(OUTPUT_DIR) / "coastal_gradient_candidate_table.csv"
summary_csv = Path(OUTPUT_DIR) / "coastal_gradient_daily_summary.csv"
gradient.to_csv(gradient_csv, index=False)
daily_summary.to_csv(summary_csv, index=False)
print("Saved:", gradient_csv)
print("Saved:", summary_csv)
display(daily_summary.head(20))


In [ ]:

fig, ax = plt.subplots(figsize=(10, 5))
if not daily_summary.empty:
    for site in ["EST", "NHV", "LWS"]:
        s = daily_summary[daily_summary["site_id"] == site].copy()
        if s.empty:
            continue
        y = s["mean_no2"] if s["mean_no2"].notna().any() else s["scenes"]
        label = f"{site} {'mean_no2' if s['mean_no2'].notna().any() else 'scene_count'}"
        ax.plot(pd.to_datetime(s["date"]), y, marker="o", label=label)
ax.set_title("Coastal gradient experiment: Eastbourne → Newhaven → Lewes")
ax.set_xlabel("Date")
ax.set_ylabel("NO2 metric (or scene count if NO2 unavailable)")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
plot_path = Path(OUTPUT_DIR) / "coastal_gradient_experiment_plot.png"
fig.savefig(plot_path, dpi=150)
plt.show()
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": "coastal_gradient",
    "sites": SITES,
    "date_from": DATE_FROM,
    "date_to": DATE_TO,
    "input_files": input_files,
    "scene_catalog_rows": int(len(cat)),
    "scene_stats_rows": int(len(stats)),
    "gradient_rows": int(len(gradient)),
    "daily_summary_rows": int(len(daily_summary)),
    "outputs": {
        "gradient_csv": str(gradient_csv),
        "summary_csv": str(summary_csv),
        "plot_png": str(plot_path),
    },
}
manifest_path = Path(OUTPUT_DIR) / "coastal_gradient_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved:", plot_path)
print("Saved:", manifest_path)
